# HW1: Frame-Level Speech Recognition

In this homework, you will be working with MFCC data consisting of 28 features at each time step/frame. Your model should be able to recognize the phoneme occured in that frame.

**Running Instructions:** This notebook is organized so that the entire experiment which yielded the best kaggle score can be run in chronological order from
* Dataset download
* Data loading
* Config Definition
* Network Architecture Definition
* Defining Model, Loss, Optimizer, and Scheduler
* Training and Validation functions
* Wandb setups
* Experiment
 * Load and Continue training
* Test and Save


# Libraries

In [ ]:
!pip install torchsummaryX==1.1.0 wandb --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 21.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 38.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 kB 45.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 14.9 MB/s eta 0:00:00


In [359]:
import torch
import numpy as np
from torchsummaryX import summary
import sklearn
import gc
import zipfile
import pandas as pd
from tqdm.auto import tqdm
import os
import datetime
import wandb
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

Device:  cuda


In [ ]:
''' If you are using colab, you can import google drive to save model checkpoints in a folder
    If you want to use it, uncomment the two lines below
'''
# from google.colab import drive
# drive.mount('/content/drive')

' If you are using colab, you can import google drive to save model checkpoints in a folder\n    If you want to use it, uncomment the two lines below\n'

In [360]:
### PHONEME LIST
PHONEMES = [
            '[SIL]',   'AA',    'AE',    'AH',    'AO',    'AW',    'AY',
            'B',     'CH',    'D',     'DH',    'EH',    'ER',    'EY',
            'F',     'G',     'HH',    'IH',    'IY',    'JH',    'K',
            'L',     'M',     'N',     'NG',    'OW',    'OY',    'P',
            'R',     'S',     'SH',    'T',     'TH',    'UH',    'UW',
            'V',     'W',     'Y',     'Z',     'ZH',    '[SOS]', '[EOS]']

In [ ]:
print(PHONEMES[39])

ZH


# Kaggle

This section contains code that helps you install kaggle's API, creating kaggle.json with you username and API key details. Make sure to input those in the given code to ensure you can download data from the competition successfully.

In [ ]:
!pip install --upgrade --force-reinstall --no-deps kaggle==1.5.8
!mkdir /root/.kaggle

with open("/root/.kaggle/kaggle.json", "w+") as f:
    f.write('{"username":"paulkotchavong","key":""}')
    # Put your kaggle username & key here

!chmod 600 /root/.kaggle/kaggle.json

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for kaggle: filename=kaggle-1.5.8-py3-none-any.whl size=73249 sha256=a068c75e612e3b2a7b9567b390195c3a1ae12e96e1e737e5f2f79d8eaf37c3cf
  Stored in directory: /root/.cache/pip/wheels/0b/76/ca/e58f8afa83166a0e68f0d5cd2e7f99d260bdc40e35da080eee
Successfully built kaggle
  Attempting uninstall: kaggle
    Found existing installation: kaggle 1.6.14
    Uninstalling kaggle-1.6.14:
      Successfully uninstalled kaggle-1.6.14


In [ ]:
# commands to download data from kaggle
!kaggle competitions download -c 11785-hw1p2-f24

!unzip -qo /content/11785-hw1p2-f24.zip -d '/content'

100% 3.98G/3.98G [00:29<00:00, 183MB/s]
100% 3.98G/3.98G [00:29<00:00, 146MB/s]


# Dataset

This section covers the dataset/dataloader class for speech data.
Below is where the Dataset class is defined. There is no further changes to the dataloader class than the default implementation which was followed from the HW1P2 write up.

In [361]:
# Dataset class to load train and validation data

class AudioDataset(torch.utils.data.Dataset):

    def __init__(self, root, phonemes = PHONEMES, context=0, partition= "train-clean-100",file_subset=None): # Feel free to add more arguments

        self.context    = context
        self.phonemes   = phonemes

        # TODO: MFCC directory - use partition to acces train/dev directories from kaggle data using root
        self.mfcc_dir       = os.path.join(root, partition, 'mfcc')
        # TODO: Transcripts directory - use partition to acces train/dev directories from kaggle data using root
        self.transcript_dir = os.path.join(root, partition, 'transcript')

        # TODO: List files in sefl.mfcc_dir using os.listdir in sorted order
        mfcc_names          = sorted(os.listdir(self.mfcc_dir))
        print(len(mfcc_names))
        # TODO: List files in self.transcript_dir using os.listdir in sorted order
        transcript_names    = sorted(os.listdir(self.transcript_dir))

        if file_subset:
            start, end = file_subset
            mfcc_names = mfcc_names[start:end]
            transcript_names = transcript_names[start:end]

        # Making sure that we have the same no. of mfcc and transcripts
        assert len(mfcc_names) == len(transcript_names)

        self.mfccs, self.transcripts = [], []

        # TODO: Iterate through mfccs and transcripts
        for i in range(len(mfcc_names)):
        #   Load a single mfcc
            mfcc        = np.load(os.path.join(self.mfcc_dir, mfcc_names[i]))
        #   Do Cepstral Normalization of mfcc (explained in writeup)
            mfcc        = (mfcc - np.mean(mfcc, axis=0, keepdims=True))/(np.std(mfcc, axis=0, keepdims=True) + 1e-5)

        #   Load the corresponding transcript
        #   Remove [SOS] and [EOS] from the transcript
            # (Is there an efficient way to do this without traversing through the transcript?)
            # Note that SOS will always be in the starting and EOS at end, as the name suggests.
            transcript = np.load(os.path.join(self.transcript_dir, transcript_names[i]))
            transcript = transcript[1:-1]
        #   Append each mfcc to self.mfcc, transcript to self.transcript
            self.mfccs.append(mfcc)
            self.transcripts.append(transcript)

        # NOTE:
        # Each mfcc is of shape T1 x 28, T2 x 28, ...
        # Each transcript is of shape (T1+2), (T2+2) before removing [SOS] and [EOS]

        # TODO: Concatenate all mfccs in self.mfccs such that
        # the final shape is T x 28 (Where T = T1 + T2 + ...)
        self.mfccs          = np.concatenate(self.mfccs, axis=0)
        print(self.mfccs.shape)

        # TODO: Concatenate all transcripts in self.transcripts such that
        # the final shape is (T,) meaning, each time step has one phoneme output
        self.transcripts    = np.concatenate(self.transcripts, axis=0)
        # print(self.transcripts[45])

        # Length of the dataset is now the length of concatenated mfccs/transcripts
        self.length = len(self.mfccs)

        # Take some time to think about what we have done.
        # self.mfcc is an array of the format (Frames x Features).
        # Our goal is to recognize phonemes of each frame
        # We can introduce context by padding zeros on top and bottom of self.mfcc
        padding = ((self.context, self.context), (0, 0))
        self.mfccs =  np.pad(self.mfccs, padding, mode='constant') # TODO

        # The available phonemes in the transcript are of string data type
        # But the neural network cannot predict strings as such.
        # Hence, we map these phonemes to integers

        # TODO: Map the phonemes to their corresponding list indexes in self.phonemes
        self.transcripts = [phonemes.index(entry) for entry in self.transcripts]
        # print(self.transcripts[45])
        # Now, if an element in self.transcript is 0, it means that it is 'SIL' (as per the above example)

    def __len__(self):
        return self.length

    def __getitem__(self, ind):

        # TODO: Based on context and offset, return a frame at given index with context frames to the left, and right.
        start = ind
        end = ind + 2*self.context + 1
        frames = self.mfccs[start:end]
        # print(self.mfccs[2])
        # print(frames)
        # After slicing, you get an array of shape 2*context+1 x 28. But our MLP needs 1d data and not 2d.
        frames = frames.flatten() # TODO: Flatten to get 1d data

        frames      = torch.FloatTensor(frames) # Convert to tensors
        phonemes    = torch.tensor(self.transcripts[ind])

        return frames, phonemes

In [362]:
class AudioTestDataset(torch.utils.data.Dataset):

    # TODO: Create a test dataset class similar to the previous class but you dont have transcripts for this
    # Imp: Read the mfccs in sorted order, do NOT shuffle the data here or in your dataloader.
    def __init__(self, root, context=0, partition= "test-clean"):
        self.context = context
        self.mfcc_dir = os.path.join(root, partition, 'mfcc')
        mfcc_names = sorted(os.listdir(self.mfcc_dir))
        self.mfccs = []
        for i in range(len(mfcc_names)):
            mfcc = np.load(os.path.join(self.mfcc_dir, mfcc_names[i]))
            mfcc = (mfcc - np.mean(mfcc, axis=0, keepdims=True))/(np.std(mfcc, axis=0, keepdims=True) + 1e-5)
            self.mfccs.append(mfcc)

        self.mfccs = np.concatenate(self.mfccs, axis=0)

        self.length = len(self.mfccs)

        padding = ((self.context, self.context), (0, 0))
        self.mfccs = np.pad(self.mfccs, padding, mode='constant')

    def __len__(self):
        return self.length

    def __getitem__(self, ind):
        start = ind
        end = ind + 2*self.context + 1
        frames = self.mfccs[start:end]
        frames = frames.flatten()
        frames = torch.FloatTensor(frames)
        return frames

# Parameters Configuration

Here I define the configuration for the specific experiment I will be running including important parameters like epoch, batch_size, initial learning rate, hidden_size, dropout, weight decay, and max learning rate. <br>

Since the best learning rate scheduler was **OneCycleLR**, mainly max learning rate was the important parameter since the scheduler automatically sets initial learning rate. <br> <br>

Among many experiments, I found that it takes at least 30 epochs on average for the model to reach pass the 85% mark whereby after learning slows down quite a bit. 40 Epochs was the most balanced as it gives the model 10 epochs at lower learning rate to squeeze out any extra learning that is left. Also, 20 context was the most appropriate as it gave enough context that doesn't violate parameter limit easily. <br> <br>

More about learning rate later in the notebook

In [363]:
config = {
    'epochs'        : 40,
    'batch_size'    : 2048,
    'context'       : 20,
    'init_lr'       : 0.01,
    'architecture'  : '7 Layers Cylinder',
    'hidden_size'   : 2048,
    'dropout'       : 0.2,
    'weight_decay'     : 1e-4,
    'optimizer'     : 'AdamW',
    'scheduler'     : 'OneCycleLR',
    'filename'      : 'mask_OneCycleLR_full_AdamW',
    'max_lr'        : 0.01,

    # Add more as you need them - e.g dropout values, weight decay, scheduler parameters
}

# Create Datasets

In [364]:
# subdata_lim = 2854 # 10% of 28539 (total number of training points)
subdata_lim = 11416 # 40% of 28539 (total number of training points)
# Create a dataset object using the AudioDataset class for the training data
# train_data = AudioDataset(root = '/content/11785-f24-hw1p2', context= config['context'], partition= "train-clean-100", file_subset=(0, subdata_lim))
train_data = AudioDataset(root = '/content/11785-f24-hw1p2', context= config['context'], partition= "train-clean-100")

# Create a dataset object using the AudioDataset class for the validation data
val_data = AudioDataset(root = '/content/11785-f24-hw1p2', context= config['context'], partition= "dev-clean")

# Create a dataset object using the AudioTestDataset class for the test data
test_data = AudioTestDataset(root = '/content/11785-f24-hw1p2', context= config['context'])

28539
(36091157, 28)
2703
(1928204, 28)


In [ ]:
# # print(val_data.transcripts.coun)
# print(val_data.transcripts[10])

# from collections import Counter
# print(Counter(val_data.transcripts))

0
Counter({0: 319908, 3: 123734, 29: 101184, 31: 97390, 23: 94541, 17: 74887, 18: 70861, 21: 65902, 9: 62763, 28: 62686, 12: 54928, 38: 54850, 6: 49332, 2: 49298, 11: 47112, 20: 47016, 22: 44728, 36: 37697, 14: 37562, 10: 37100, 13: 36184, 16: 34813, 27: 34131, 25: 30755, 1: 29688, 4: 29340, 35: 27440, 34: 26691, 7: 23607, 5: 20274, 24: 19327, 30: 17628, 15: 13541, 8: 12644, 37: 9669, 32: 9247, 19: 8730, 33: 6286, 26: 3861, 39: 869})


In [365]:
# Define dataloaders for train, val and test datasets
# Dataloaders will yield a batch of frames and phonemes of given batch_size at every iteration
# We shuffle train dataloader but not val & test dataloader. Why?

train_loader = torch.utils.data.DataLoader(
    dataset     = train_data,
    num_workers = 4,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = True
)

val_loader = torch.utils.data.DataLoader(
    dataset     = val_data,
    num_workers = 2,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = False
)

test_loader = torch.utils.data.DataLoader(
    dataset     = test_data,
    num_workers = 2,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = False
)


print("Batch size     : ", config['batch_size'])
print("Context        : ", config['context'])
print("Input size     : ", (2*config['context']+1)*28)
print("Output symbols : ", len(PHONEMES))

print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Validation dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

Batch size     :  2048
Context        :  20
Input size     :  1148
Output symbols :  42
Train dataset samples = 36091157, batches = 17623
Validation dataset samples = 1928204, batches = 942
Test dataset samples = 1934138, batches = 945


In [366]:
# Testing code to check if your data loaders are working
for i, data in enumerate(train_loader):
    frames, phoneme = data
    print(frames.shape, phoneme.shape)
    break

torch.Size([2048, 1148]) torch.Size([2048])


# Network Architecture


This section defines your network architecture for the homework. Throughout the assignment, I've tried cylinder architecture and variations of pyramid architecture (through modifying how aggressive the width reduces as the model approaches output layer). <br> <br>


The best architecture is given below. It is a pyramid architecture that is not as aggressive (input layer goes from 1140 input size -> 2048 hidden size -> ultimately reduced to 1696 hidden size before output layer sized 42) <br> <br>


Instead of implementing the masking in the dataset class, I chose to take advice of implementing it in the forward pass utilizing torchaudio.transforms. I am doing both frequency and time masking every pass to attempt to reduce overfitting. <br> <br>

Finally, I also implemented weight initialization based on a normal distribution because in some runs, training crashed, likely due to exploding gradient.

In [367]:
# This architecture will make you cross the very low cutoff
# However, you need to run a lot of experiments to cross the medium or high cutoff
import torch
import torchaudio.transforms as T

class Network(torch.nn.Module):
    def __init__(self, input_size, output_size, hidden_size, time_mask_param, freq_mask_param, context, apply_masking = False, training = False):
        super(Network, self).__init__()
        self.time_mask = T.TimeMasking(time_mask_param=time_mask_param,iid_masks=True)
        self.freq_mask = T.FrequencyMasking(freq_mask_param=freq_mask_param,iid_masks=True)
        self.apply_masking = apply_masking
        self.context = context
        self.training = training

        self.model = torch.nn.Sequential(
            torch.nn.Linear(input_size, hidden_size),
            torch.nn.BatchNorm1d(hidden_size),
            torch.nn.GELU(),
            torch.nn.Dropout(config['dropout']),

            torch.nn.Linear(hidden_size, int(0.963**1*hidden_size)),
            torch.nn.BatchNorm1d(int(0.963**1*hidden_size)),
            torch.nn.GELU(),
            torch.nn.Dropout(config['dropout']),

            torch.nn.Linear(int(0.963**1*hidden_size), int(0.963**2*hidden_size)),
            torch.nn.BatchNorm1d(int(0.963**2*hidden_size)),
            torch.nn.GELU(),
            torch.nn.Dropout(config['dropout']),

            torch.nn.Linear(int(0.963**2*hidden_size), int(0.963**3*hidden_size)),
            torch.nn.BatchNorm1d(int(0.963**3*hidden_size)),
            torch.nn.GELU(),
            torch.nn.Dropout(config['dropout']),

            torch.nn.Linear(int(0.963**3*hidden_size), int(0.963**4*hidden_size)),
            torch.nn.BatchNorm1d(int(0.963**4*hidden_size)),
            torch.nn.GELU(),
            torch.nn.Dropout(config['dropout']),

            torch.nn.Linear(int(0.963**4*hidden_size), int(0.963**5*hidden_size)),
            torch.nn.BatchNorm1d(int(0.963**5*hidden_size)),
            torch.nn.GELU(),
            torch.nn.Dropout(config['dropout']),

            torch.nn.Linear(int(0.963**5*hidden_size), output_size)
        )

        # Initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        # He initialization for all linear layers in the model
        for layer in self.model:
            if isinstance(layer, torch.nn.Linear):
                fan_in = layer.weight.size(1)
                torch.nn.init.normal_(layer.weight, mean=0, std=np.sqrt(1.0 / fan_in))
                # torch.nn.init.xavier_uniform_(layer.weight)
                torch.nn.init.zeros_(layer.bias)

    def forward(self, x):
      current_batch_size = x.size(0)
      if self.training and self.apply_masking:
        x = x.view(current_batch_size, 28, 2 * self.context + 1)
        x = self.time_mask(x)
        x = self.freq_mask(x)
        x = x.reshape(current_batch_size, -1)  # Flatten again back to 1D
      return self.model(x)


# Define Model, Loss Function and Optimizer

Here we define the model, loss function, optimizer and optionally a learning rate scheduler.

In [368]:
INPUT_SIZE  = (2*config['context'] + 1) * 28 # Why is this the case?
print(INPUT_SIZE)
model       = Network(INPUT_SIZE, len(train_data.phonemes), config['hidden_size'],time_mask_param=5,freq_mask_param=3,context=config['context'],apply_masking=False).to(device)
summary(model, frames.to(device))
# Check number of parameters of your network
# Remember, you are limited to 20 million parameters for HW1 (including ensembles)

1148
----------------------------------------------------------------------------------------------------
Layer                   Kernel Shape         Output Shape         # Params (K)      # Mult-Adds (M)
0_Linear                [1148, 2048]         [2048, 2048]             2,353.15                 2.35
1_BatchNorm1d                 [2048]         [2048, 2048]                 4.10                 0.00
2_GELU                             -         [2048, 2048]                    -                    -
3_Dropout                          -         [2048, 2048]                    -                    -
4_Linear                [2048, 1972]         [2048, 1972]             4,040.63                 4.04
5_BatchNorm1d                 [1972]         [2048, 1972]                 3.94                 0.00
6_GELU                             -         [2048, 1972]                    -                    -
7_Dropout                          -         [2048, 1972]                    -                

The main takeaway from below is the learning rate scheduler. As mentioned before OneCycleLR was the scheduler that was used to train the majority of the experiments and gave the best result. The reason it was chosen is because it had a smooth warmup period until max learning rate and start to anneal down with cosine strategy. <br> <br>

I experimented with div_factor (which influences how fast/slow the learning warmups) and max learning rate. I found that the best warmup period is about 10 epochs with max learning rate of 0.01. I theoized that higher max learning rate than 0.01 would allow the model to learn more as it anneals down from a higher max learning rate. I also tried lower than 0.01 learning rate, but ultimately found that 0.01 was the best.

In [369]:
criterion = torch.nn.CrossEntropyLoss() # Defining Loss function.
# We use CE because the task is multi-class classification

optimizer = torch.optim.AdamW(model.parameters(), lr= config['init_lr'], weight_decay=config['weight_decay']) #Defining Optimizer
# Recommended : Define Scheduler for Learning Rate,
# including but not limited to StepLR, MultiStep, CosineAnnealing, CosineAnnealingWithWarmRestarts, ReduceLROnPlateau, etc.
# You can refer to Pytorch documentation for more information on how to use them.
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=1)
# warmup_epochs = 3
# scheduler = WarmupCosineAnnealingLR(optimizer, warmup_epochs, config['epochs'])
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer,
  max_lr=config['max_lr'],
  steps_per_epoch=len(train_loader),
  epochs=int(config['epochs']),
  pct_start=0.3,
  div_factor=10,
  final_div_factor=1000)

In [ ]:
checkpoint = torch.load('OneCycleLR_full_AdamW-model_epoch_19.pth')

# Load the saved state dicts
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

# Training and Validation Functions

This section covers the training, and validation functions for each epoch of running your experiment with a given model architecture.

In [370]:
torch.cuda.empty_cache()
gc.collect()

4252

In [372]:
def train(model, dataloader, optimizer, criterion, scheduler=None):

    model.train()
    tloss, tacc = 0, 0 # Monitoring loss and accuracy
    batch_bar   = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')

    scaler = torch.cuda.amp.GradScaler()
    for i, (frames, phonemes) in enumerate(dataloader):

        ### Initialize Gradients
        optimizer.zero_grad()

        ### Move Data to Device (Ideally GPU)
        frames      = frames.to(device)
        phonemes    = phonemes.to(device)
        with torch.cuda.amp.autocast():
            ### Forward Propagation
            logits  = model(frames)

            ### Loss Calculation
            loss    = criterion(logits, phonemes)

        ### Backward Propagation
        scaler.scale(loss).backward()

        # ### Gradient Clipping (inserted here, before optimizer step)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)

        ### Update Weights
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
          scheduler.step()

        tloss   += loss.item()
        tacc    += torch.sum(torch.argmax(logits, dim= 1) == phonemes).item()/logits.shape[0]

        batch_bar.set_postfix(loss="{:.04f}".format(float(tloss / (i + 1))),
                              acc="{:.04f}%".format(float(tacc*100 / (i + 1))))
        batch_bar.update()

        ### Release memory
        del frames, phonemes, logits
        torch.cuda.empty_cache()

    batch_bar.close()
    tloss   /= len(train_loader)
    tacc    /= len(train_loader)

    return tloss, tacc

In [ ]:
def eval(model, dataloader):

    model.eval() # set model in evaluation mode
    vloss, vacc = 0, 0 # Monitoring loss and accuracy
    batch_bar   = tqdm(total=len(val_loader), dynamic_ncols=True, position=0, leave=False, desc='Val')

    for i, (frames, phonemes) in enumerate(dataloader):

        ### Move data to device (ideally GPU)
        frames      = frames.to(device)
        phonemes    = phonemes.to(device)

        # makes sure that there are no gradients computed as we are not training the model now
        with torch.inference_mode():
            ### Forward Propagation
            logits  = model(frames)
            ### Loss Calculation
            loss    = criterion(logits, phonemes)

        vloss   += loss.item()
        vacc    += torch.sum(torch.argmax(logits, dim= 1) == phonemes).item()/logits.shape[0]

        # Do you think we need loss.backward() and optimizer.step() here?

        batch_bar.set_postfix(loss="{:.04f}".format(float(vloss / (i + 1))),
                              acc="{:.04f}%".format(float(vacc*100 / (i + 1))))
        batch_bar.update()

        ### Release memory
        del frames, phonemes, logits
        torch.cuda.empty_cache()

    batch_bar.close()
    vloss   /= len(val_loader)
    vacc    /= len(val_loader)

    return vloss, vacc

# Weights and Biases Setup

This section is to enable logging metrics and files with Weights and Biases. Please refer to wandb documentationa and recitation 0 that covers the use of weights and biases for logging, hyperparameter tuning and monitoring your runs for your homeworks. Using this tool makes it very easy to show results when submitting your code and models for homeworks, and also extremely useful for study groups to organize and run ablations under a single team in wandb.

We have written code for you to make use of it out of the box, so that you start using wandb for all your HWs from the beginning.

In [381]:
wandb.login(key="0cc2199d5dbab5108d50abc33d06f093f13e6a18") #API Key is in your wandb account, under settings (wandb.ai/settings)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [ ]:
# Create your wandb run
run = wandb.init(
    name    = f"{config['filename']}-run", ### Wandb creates random run names if you skip this field, we recommend you give useful names
    reinit  = True, ### Allows reinitalizing runs when you re-run this cell
    # id     = "73cljq90", ### Insert specific run id here if you want to resume a previous run
    # resume = "must", ### You need this to resume previous runs, but comment out reinit = True when using this
    project = "hw1p2", ### Project should be created in your wandb account
    config  = config ### Wandb Config for your run
)

lr,▁▁▁▂▃▄▅▅▆▇███████▇▇▇▇▆▆▆▅▅▄▄
train_acc,▁▃▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇████████
train_loss,█▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▆▇▇██████████████▇▇▇▇▇▇▇▇▇
valid_loss,▆▃▂▁▁▁▁▁▂▂▂▃▃▄▄▄▄▅▅▆▆▆▆▇▇▇██
lr,0.00044
train_acc,91.99921
train_loss,0.21552
val_acc,80.89825
valid_loss,0.7994


In [ ]:
### Save your model architecture as a string with str(model)
model_arch  = str(model)

### Save it in a txt file
arch_file   = open(f"{config['filename']}-model_arch.txt", "w")
file_write  = arch_file.write(model_arch)
arch_file.close()

### log it in your wandb run with wandb.save()
wandb.save(f"{config['filename']}-model_arch.txt")

['/content/wandb/run-20240922_171634-g27f4pq5/files/back2basic_NoMask_OneCycleLR_full_AdamW-model_arch.txt']

# Experiment

This section is the main training loop which runs the experiment. Wandb should logging metrics every epoch and the best model is also saved at each epoch if accuracy improves

In [ ]:
# Iterate over number of epochs to train and evaluate your model
torch.cuda.empty_cache()
gc.collect()
wandb.watch(model, log="all")

curr_best = -1 # Initialize best accuracy to -1

for epoch in range(config['epochs']):

    print("\nEpoch {}/{}".format(epoch+1, config['epochs']))

    curr_lr                 = float(optimizer.param_groups[0]['lr'])
    model.training = True
    train_loss, train_acc   = train(model, train_loader, optimizer, criterion, scheduler)
    model.traing = False
    val_loss, val_acc       = eval(model, val_loader)

    # scheduler.step(val_loss)

    print("\tTrain Acc {:.04f}%\tTrain Loss {:.04f}\t Learning Rate {:.07f}".format(train_acc*100, train_loss, curr_lr))
    print("\tVal Acc {:.04f}%\tVal Loss {:.04f}".format(val_acc*100, val_loss))

    ### Log metrics at each epoch in your run
    # Optionally, you can log at each batch inside train/eval functions
    # (explore wandb documentation/wandb recitation)
    wandb.log({'train_acc': train_acc*100, 'train_loss': train_loss,
               'val_acc': val_acc*100, 'valid_loss': val_loss, 'lr': curr_lr})

    ### Highly Recommended: Save checkpoint in drive and/or wandb if accuracy is better than your current best
    if val_acc > curr_best:
        print("Best validation accuracy improved from {} to {}".format(curr_best, val_acc))
        curr_best = val_acc
        # save model to file
        checkpoint = {
          'epoch': epoch + 1,
          'model_state_dict': model.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'scheduler_state_dict': scheduler.state_dict(),
          'loss': val_loss
        }
        torch.save(checkpoint, f"{config['filename']}-model_epoch_{epoch+1}.pth")



## Reload current best model and retrain


In this section, I loaded any model that performed well and attempted to train the model further, whether by choosing a different learning rate and strategy, etc.
**Note:** Here I loaded my model below, but after attempting training for 10 more epoch and it didn't quite improve. Therefore, I submitted my prediction using the checkpoint that was originally loaded for this training

* First we redefine the config as necessary

In [382]:
config = {
    'epochs'        : 10,
    'batch_size'    : 2048,
    'context'       : 20,
    'init_lr'       : 0.01,
    'architecture'  : '7 Layers Cylinder',
    'hidden_size'   : 2048,
    'dropout'       : 0.2,
    'weight_decay'     : 1e-4,
    'optimizer'     : 'AdamW',
    'scheduler'     : 'OneCycleLR',
    'filename'      : 'further_mask_OneCycleLR_full_AdamW',
    'max_lr'        : 0.01,
}



*   Then, we load the weight from previous training iteration and reset the weights of optimizer for CosineAnnealing strategy




In [374]:
checkpoint = torch.load('mask_mask_OneCycleLR_full_AdamW-model_epoch_38.pth')

# Load the saved state dicts
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
for param_group in optimizer.param_groups:
    param_group['lr'] = 0.01

# scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_0=5,eta_min=2e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=1e-6)
# scheduler.load_state_dict(checkpoint['scheduler_state_dict'])




*   Specify the wandb run to continue




In [ ]:
# Create your wandb run
run = wandb.init(
    name    = f"{config['filename']}-run", ### Wandb creates random run names if you skip this field, we recommend you give useful names
    # reinit  = True, ### Allows reinitalizing runs when you re-run this cell
    id     = "smaz06jc", ### Insert specific run id here if you want to resume a previous run
    resume = "must", ### You need this to resume previous runs, but comment out reinit = True when using this
    project = "hw1p2", ### Project should be created in your wandb account
    config  = config ### Wandb Config for your run
)



*   Identical training loop as previous, but ported over for sequential running purposes.




In [ ]:
# Iterate over number of epochs to train and evaluate your model
torch.cuda.empty_cache()
gc.collect()
wandb.watch(model, log="all")

curr_best = -1 # Initialize best accuracy to -1

for epoch in range(1,config['epochs']):

    print("\nEpoch {}/{}".format(epoch+1, config['epochs']))

    curr_lr                 = float(optimizer.param_groups[0]['lr'])
    model.training = True
    train_loss, train_acc   = train(model, train_loader, optimizer, criterion, scheduler)
    model.traing = False
    val_loss, val_acc       = eval(model, val_loader)

    # scheduler.step(val_loss)

    print("\tTrain Acc {:.04f}%\tTrain Loss {:.04f}\t Learning Rate {:.07f}".format(train_acc*100, train_loss, curr_lr))
    print("\tVal Acc {:.04f}%\tVal Loss {:.04f}".format(val_acc*100, val_loss))

    ### Log metrics at each epoch in your run
    # Optionally, you can log at each batch inside train/eval functions
    # (explore wandb documentation/wandb recitation)
    wandb.log({'train_acc': train_acc*100, 'train_loss': train_loss,
               'val_acc': val_acc*100, 'valid_loss': val_loss, 'lr': curr_lr})

    ### Highly Recommended: Save checkpoint in drive and/or wandb if accuracy is better than your current best
    if val_acc > curr_best:
        print("Best validation accuracy improved from {} to {}".format(curr_best, val_acc))
        curr_best = val_acc
        # save model to file
        checkpoint = {
          'epoch': epoch + 1,
          'model_state_dict': model.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'scheduler_state_dict': scheduler.state_dict(),
          'loss': val_loss
        }
        torch.save(checkpoint, f"{config['filename']}-model_epoch_{epoch+1}.pth")

The above continuation of the training was done by resetting learning rate for each param group to 0.01 (the max learning rate) and continue to cosine anneal down for another 10 epoch, but not a significant improvement was found. As a result, we used the best weight for prediction.

# Testing and submission to Kaggle

Before we get to the following code, make sure to see the format of submission given in *sample_submission.csv*. Once you have done so, it is time to fill the following function to complete your inference on test data. Refer the eval function from previous cells to get an idea of how to go about completing this function.

In [375]:
def test(model, test_loader):
    ### What you call for model to perform inference?
    model.eval() # TODO train or eval?

    ### List to store predicted phonemes of test data
    test_predictions = []

    ### Which mode do you need to avoid gradients?
    with torch.no_grad(): # TODO

        for i, mfccs in enumerate(tqdm(test_loader)):

            mfccs   = mfccs.to(device)

            logits  = model(mfccs)

            ### Get most likely predicted phoneme with argmax
            predicted_phonemes = torch.argmax(logits, dim=1)

            ### How do you store predicted_phonemes with test_predictions? Hint, look at eval
            # TODO
            predicted_classes = [PHONEMES[i.item()] for i in predicted_phonemes]
            test_predictions.extend(predicted_classes)

    return test_predictions

In [376]:
model.load_state_dict(torch.load('mask_mask_OneCycleLR_full_AdamW-model_epoch_38.pth')['model_state_dict'])
model.eval()
predictions = test(model, test_loader)

  0%|          | 0/945 [00:00<?, ?it/s]

In [377]:
### Create CSV file with predictions
with open("./submission_recent_full_testing.csv", "w+") as f:
    f.write("id,label\n")
    for i in range(len(predictions)):
        f.write("{},{}\n".format(i, predictions[i]))

* save model weight to wandb

In [ ]:
wandb.save('mask_mask_OneCycleLR_full_AdamW-model_epoch_38.pth')

In [ ]:
### Finish your wandb run
run.finish()

In [ ]:
### Submit to kaggle competition using kaggle API (Uncomment below to use)
# !kaggle competitions submit -c 11785-hw1p2-f24 -f ./submission.csv -m "Test Submission"

### However, its always safer to download the csv file and then upload to kaggle